# Pydantic AI Hosted Agent Setup

This notebook helps you:
1. Build and push a container image for your Pydantic AI hosted agent code.
2. Create or update a hosted agent version from Python.
3. Grant Foundry role access to the hosted agent identity.

## Prerequisites
- Your signed-in identity must have `Container Registry Repository Writer` on the target ACR repository (or equivalent push permission).
- The Azure Foundry project/resource identity must have pull access to the image in ACR (for example `Container Registry Repository Reader` or equivalent repository-level pull permission).

Fill in all placeholder values before running code cells.

## 1. Build And Push The Container Image

This runs Azure Container Registry cloud build from the current pydantic-ai folder.

In [ ]:
# Set these placeholders
ACR_NAME = "<your-acr-name>"
IMAGE_NAME = "<your-image-name>"
IMAGE_TAG = "<your-image-tag>"

print(f"ACR_NAME={ACR_NAME}")
print(f"IMAGE_NAME={IMAGE_NAME}")
print(f"IMAGE_TAG={IMAGE_TAG}")

In [ ]:
import subprocess

build_cmd = [
    "az", "acr", "build",
    "-t", f"{IMAGE_NAME}:{IMAGE_TAG}",
    "-r", ACR_NAME,
    "--source-acr-auth-id", "[caller]",
    ".",
]

print("Running:", " ".join(build_cmd))
subprocess.run(build_cmd, check=True)

## 2. Create Hosted Agent Version

Use this cell to register your hosted agent container image in Foundry.

Install dependencies first if needed:
`pip install azure-ai-projects azure-identity`

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    HostedAgentDefinition,
    ProtocolVersionRecord,
    AgentProtocol,
    ContainerConfiguration,
)
from azure.identity import DefaultAzureCredential

# Placeholder values - update before running
PROJECT_ENDPOINT = "https://<your-project-endpoint>.services.ai.azure.com/api/projects/<your-project-name>"
AGENT_NAME = "<your-agent-name>"

ACR_LOGIN_SERVER = "<your-acr-name>.azurecr.io"
IMAGE_NAME = "<your-image-name>"
IMAGE_TAG = "<your-image-tag>"
IMAGE_URI = f"{ACR_LOGIN_SERVER}/{IMAGE_NAME}:{IMAGE_TAG}"

# Environment variables passed to your containerized Pydantic AI app
AZURE_OPENAI_ENDPOINT = "https://<your-openai-resource>.openai.azure.com/"
AZURE_OPENAI_API_VERSION = "<api-version>"
AZURE_OPENAI_DEPLOYMENT = "<deployment-name>"

credential = DefaultAzureCredential()
project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
    allow_preview=True,
)

agent = project.agents.create_version(
    agent_name=AGENT_NAME,
    definition=HostedAgentDefinition(
        protocol_versions=[
            ProtocolVersionRecord(protocol=AgentProtocol.RESPONSES, version="1.0.0")
        ],
        cpu="1",
        memory="2Gi",
        container_configuration=ContainerConfiguration(image=IMAGE_URI),
        environment_variables={
            "AZURE_OPENAI_ENDPOINT": AZURE_OPENAI_ENDPOINT,
            "AZURE_OPENAI_API_VERSION": AZURE_OPENAI_API_VERSION,
            "AZURE_OPENAI_DEPLOYMENT": AZURE_OPENAI_DEPLOYMENT,
            "LOG_LEVEL": "INFO",
        },
    ),
)

print(f"Agent created: {agent.name}, version: {agent.version}")

## 3. Grant Foundry User Role To The Hosted Agent Identity

After the hosted agent version is created, Azure provisions an agent identity (service principal/object ID).

Grant that identity the `Foundry User` role on your Foundry resource so it can access Foundry capabilities at runtime.

In [ ]:
import subprocess

# Placeholders - update these values
AGENT_IDENTITY_OBJECT_ID = "<hosted-agent-identity-object-id>"
FOUNDRY_RESOURCE_ID = "/subscriptions/<sub-id>/resourceGroups/<rg-name>/providers/Microsoft.CognitiveServices/accounts/<foundry-resource-name>"
ROLE_NAME = "Foundry User"

assign_cmd = [
    "az", "role", "assignment", "create",
    "--assignee-object-id", AGENT_IDENTITY_OBJECT_ID,
    "--assignee-principal-type", "ServicePrincipal",
    "--role", ROLE_NAME,
    "--scope", FOUNDRY_RESOURCE_ID,
]

print("Running:", " ".join(assign_cmd))
subprocess.run(assign_cmd, check=True)